# Baseline anomaly modeling

FORGE | Training-data preparation

The baseline detector learns from observations designated as normal in the training partition. This notebook verifies the source files and selects those observations, then removes duplicates to produce a table of eight sensor measurements. It also examines the pressure channel's spread and displays saved benchmark results.

Only training recordings are loaded. Reusable detector fitting and validation are implemented in the package.


## Training recording selection

The [split manifest](../data/skab-splits.json) assigns recording IDs to training, validation, and testing. Resolving the project root through the installed FORGE package allows the notebook to find this file regardless of its working directory.

The training partition contains 14 recording IDs. This first step identifies the files. Loading their measurements and selecting normal observations follow below. The [split protocol](../docs/data-split.md) explains the allocation and overlap rules.


In [ ]:
import json

from forge.config import PROJECT_ROOT

split_path = PROJECT_ROOT / "data" / "skab-splits.json"
split_plan = json.loads(split_path.read_text(encoding="utf-8"))
train_ids = split_plan["partitions"]["train"]

print(train_ids)
print(f"Training recordings: {len(train_ids)}")

## Recording lookup and split checks

Each training ID is matched to its inventory entry in the order given by the split manifest. Before opening a CSV, the validator checks that every recording has an assignment, overlapping recordings stay together, and the inspected development recording remains in training. The inventory supplies file paths, checksums, and annotation availability.


In [ ]:
import numpy as np
import pandas as pd

from forge.data.partitions import inspect_recording, recording_path, validate_plan
from forge.data.sample import SENSOR_UNITS

inventory_path = PROJECT_ROOT / "data" / "skab-inventory.json"
inventory = json.loads(inventory_path.read_text(encoding="utf-8"))
validate_plan(inventory, split_plan)

records_by_id = {record["experiment_id"]: record for record in inventory["records"]}
train_records = [records_by_id[experiment_id] for experiment_id in train_ids]
sensor_columns = list(SENSOR_UNITS)

training_files = pd.DataFrame(
    {
        "experiment_id": record["experiment_id"],
        "local_path": record["local_path"],
        "annotations_present": record["annotations_present"],
        "file_exists": recording_path(record).is_file(),
    }
    for record in train_records
)
print(training_files.to_string(index=False))

## Verified training measurements

The loader checks each file against its checksum and inventory metadata. It preserves timestamps and sampling gaps, and keeps the original recordings separate in `train_frames`. Their source annotations and file boundaries remain available for later inspection.


In [ ]:
train_frames = {record["experiment_id"]: inspect_recording(record) for record in train_records}
raw_training_rows = sum(len(frame) for frame in train_frames.values())
print(f"Verified training recordings: {len(train_frames)}")
print(f"Recorded rows before selection: {raw_training_rows:,}")

## Normal-observation selection

For annotated recordings, only observations with `anomaly == 0` enter the training pool. All rows from `anomaly-free/anomaly-free` are included because the source catalog designates that recording as normal. The file remains without annotation columns.

Each selected observation keeps its timestamp, experiment ID, source row position, and basis for normal selection. These fields provide provenance separately from the model inputs. Normal status follows the dataset's designation and has not been independently verified against the equipment's condition.


In [ ]:
normal_parts = []
selection_counts = []

for record in train_records:
    experiment_id = record["experiment_id"]
    frame = train_frames[experiment_id]

    if record["annotations_present"]:
        selected = frame.loc[frame["anomaly"].eq(0)]
        normal_basis = "source_annotation"
    elif experiment_id == "anomaly-free/anomaly-free":
        selected = frame
        normal_basis = "source_catalog"
    else:
        raise ValueError(f"No normal-selection rule for {experiment_id}")

    normal_rows = selected.loc[:, ["datetime", *sensor_columns]].copy()
    normal_rows["experiment_id"] = experiment_id
    normal_rows["source_row"] = selected.index
    normal_rows["normal_basis"] = normal_basis
    normal_parts.append(normal_rows)
    selection_counts.append(
        {
            "experiment_id": experiment_id,
            "recorded_rows": len(frame),
            "normal_rows": len(selected),
            "excluded_rows": len(frame) - len(selected),
            "normal_basis": normal_basis,
        }
    )

normal_training_rows = pd.concat(normal_parts, ignore_index=True)
selection_summary = pd.DataFrame(selection_counts)
print(selection_summary.to_string(index=False))
print(f"Selected normal observations: {len(normal_training_rows):,}")

## Repeated-observation removal

Observations count as duplicates when their timestamp and all eight sensor values match. Equal measurements recorded at different times are retained. The first occurrence in manifest order enters the training table, while `duplicate_training_rows` keeps discarded occurrences and their provenance.

This removes the overlap between the source-normal recording and `other/5`. File boundaries and source row positions remain available for inspection. The pooled table supports an initial model that scores individual observations. Any later temporal windows must be built within the original recordings.


In [ ]:
observation_key = ["datetime", *sensor_columns]
duplicate_mask = normal_training_rows.duplicated(subset=observation_key, keep="first")
duplicate_training_rows = normal_training_rows.loc[duplicate_mask].copy()
training_rows = normal_training_rows.loc[~duplicate_mask].reset_index(drop=True)

print(f"Repeated observations removed: {len(duplicate_training_rows):,}")
print(f"Unique normal observations retained: {len(training_rows):,}")

## Sensor feature table

`X_train` contains the eight sensor measurements in a fixed column order and their original units. Aligned timestamps and source identifiers are kept separately in `training_provenance`. The feature table excludes these provenance fields and the source annotations.

The checks below reject empty or non-finite input and repeated observations in the retained data. The following cells examine the training medians and pressure spread. Full detector fitting is handled by the package.


In [ ]:
X_train = training_rows.loc[:, sensor_columns].copy()
training_provenance = training_rows.loc[
    :, ["datetime", "experiment_id", "source_row", "normal_basis"]
].copy()

assert not X_train.empty, "No normal training observations were selected."
assert X_train.columns.to_list() == sensor_columns
assert X_train.index.equals(training_provenance.index)
assert np.isfinite(X_train.to_numpy()).all(), "Training features contain non-finite values."
assert not training_rows.duplicated(subset=observation_key).any()

print(f"X_train shape: {X_train.shape}")
print(f"Feature columns: {X_train.columns.to_list()}")
print("No preprocessing or detector has been fitted.")

## Preparation results

| Stage | Observations |
|---|---:|
| Recorded rows in 14 training files | 23,893 |
| Anomalous rows excluded | 5,015 |
| Normal observations before deduplication | 18,878 |
| Repeated normal observations removed | 572 |
| Unique normal observations retained | 18,306 |

The resulting `X_train` contains 18,306 rows and eight sensor columns. The source-normal recording contributes 9,405 rows, so the fitting data contains an uneven contribution from each recording. This imbalance needs to be considered during model development.

These counts describe the prepared data, and the notebook leaves validation and test CSVs unopened. The package fits the statistical baseline and learned detector, then selects their thresholds on validation data. The final cell displays the saved benchmark aggregates.


In [ ]:
train_medians = X_train.median()
print(train_medians)

In [ ]:
absolute_deviations = X_train.sub(train_medians, axis="columns").abs()
train_mad = absolute_deviations.median()
print(train_mad)

In [ ]:
pressure = X_train["Pressure"]

print("Exact pressure MAD:", repr(train_mad["Pressure"]))
print("Distinct pressure values:", pressure.nunique())
print("Most frequent values:")
print(pressure.value_counts().head())

In [ ]:
pressure_quartiles = pressure.quantile([0.25, 0.75])
pressure_iqr = pressure_quartiles.loc[0.75] - pressure_quartiles.loc[0.25]

print(pressure_quartiles)
print("Pressure IQR:", pressure_iqr)

In [ ]:
import json
from pathlib import Path
import pandas as pd

results_path = PROJECT_ROOT / "docs" / "evaluation-results.json"
results = json.loads(results_path.read_text(encoding="utf-8"))

per_group = results["test"]["metrics"]["per_group"]

comparison = pd.DataFrame.from_dict(per_group, orient="index")
comparison["false_positive_rate"] = (comparison["fp"]/(comparison["fp"] + comparison["tn"]))

comparison[
    ["tn", "fp", "fn", "tp", "precision", "recall", "f1", "false_positive_rate"]
].sort_values("false_positive_rate", ascending=False)